# Practical No: 3
**Subject:** Deep Learning

**Problem Statement:** Implement forward propagation and backpropagation using TensorFlow/Keras. Analyze the effect of different learning rates and the number of epochs on model performance.

**Dataset:** MNIST (via KaggleHub)

In [ ]:
!pip install kagglehub -q

In [ ]:
import kagglehub
import numpy as np
import os, struct, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
path = kagglehub.dataset_download("hojjatk/mnist-dataset")
print("Path to dataset files:", path)

In [ ]:
all_files = []
for root, dirs, fnames in os.walk(path):
    for fn in fnames:
        all_files.append(os.path.join(root, fn))

print("Files found:")
for f in all_files:
    print(" ", f)

def find_file(keyword):
    candidates = [f for f in all_files if keyword in os.path.basename(f).lower()]
    if not candidates:
        raise FileNotFoundError(f"No file matched keyword: {keyword}")
    candidates.sort(key=len)  # prefer shortest path -> avoids nested duplicates
    return candidates[0]

def read_idx_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
    return data

def read_idx_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

train_images_path = find_file("train-images")
train_labels_path = find_file("train-labels")
test_images_path  = find_file("t10k-images")
test_labels_path  = find_file("t10k-labels")

print("\nUsing:")
print(" train images:", train_images_path)
print(" train labels:", train_labels_path)
print(" test images :", test_images_path)
print(" test labels :", test_labels_path)

x_train = read_idx_images(train_images_path)
y_train = read_idx_labels(train_labels_path)
x_test  = read_idx_images(test_images_path)
y_test  = read_idx_labels(test_labels_path)

print("\nTrain:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat  = x_test.reshape(-1, 28*28)

print("Flattened train shape:", x_train_flat.shape)
print("Flattened test shape :", x_test_flat.shape)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST digits")
plt.tight_layout()
plt.show()

# Class distribution
plt.figure(figsize=(7,4))
sns.countplot(x=y_train)
plt.title("Training Set Class Distribution")
plt.xlabel("Digit"); plt.ylabel("Count")
plt.show()

In [ ]:
def build_model(lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation="relu", name="hidden1"),
        layers.Dense(64, activation="relu", name="hidden2"),
        layers.Dense(10, activation="softmax", name="output")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model(lr=0.001)
model.summary()

## Manual forward + backward pass (explicit)

In [ ]:
tf.keras.backend.clear_session()
manual_model = build_model(lr=0.001)
optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn = keras.losses.SparseCategoricalCrossentropy()

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        # ---- FORWARD PASS ----
        predictions = manual_model(x_batch, training=True)
        loss = loss_fn(y_batch, predictions)
    # ---- BACKWARD PASS (backpropagation) ----
    gradients = tape.gradient(loss, manual_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, manual_model.trainable_variables))
    acc = tf.reduce_mean(
        tf.cast(tf.equal(tf.argmax(predictions, axis=1),
                          tf.cast(y_batch, tf.int64)), tf.float32))
    return loss, acc, gradients

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((x_train_flat, y_train))
dataset = dataset.shuffle(10000).batch(batch_size)

manual_losses, manual_accs, grad_norms = [], [], []
print("Manual training loop (3 epochs) \u2013 explicit forward + backprop:")
for epoch in range(3):
    epoch_loss, epoch_acc, n_batches = 0, 0, 0
    for x_batch, y_batch in dataset:
        loss, acc, grads = train_step(x_batch, y_batch)
        epoch_loss += loss.numpy(); epoch_acc += acc.numpy(); n_batches += 1
    grad_norm = tf.linalg.global_norm(grads).numpy()
    manual_losses.append(epoch_loss / n_batches)
    manual_accs.append(epoch_acc / n_batches)
    grad_norms.append(grad_norm)
    print(f"Epoch {epoch+1}: loss={manual_losses[-1]:.4f}, "
          f"acc={manual_accs[-1]:.4f}, grad_norm={grad_norm:.4f}")

plt.figure(figsize=(6,4))
plt.plot(range(1, len(grad_norms)+1), grad_norms, marker="o", color="crimson")
plt.title("Gradient Norm per Epoch (Backprop Signal Strength)")
plt.xlabel("Epoch"); plt.ylabel("Global Gradient Norm")
plt.grid(alpha=0.3)
plt.show()

## Experiment with different learning rates and epochs

In [ ]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]
epoch_options  = [5, 15, 30]

results = []
histories = {}

for lr in learning_rates:
    for ep in epoch_options:
        print(f"Training: lr={lr}, epochs={ep}")
        tf.keras.backend.clear_session()
        m = build_model(lr=lr)
        start = time.time()
        h = m.fit(x_train_flat, y_train,
                  validation_split=0.1,
                  epochs=ep, batch_size=256, verbose=0)
        elapsed = time.time() - start
        test_loss, test_acc = m.evaluate(x_test_flat, y_test, verbose=0)
        results.append({
            "lr": lr, "epochs": ep,
            "train_acc": h.history["accuracy"][-1],
            "val_acc": h.history["val_accuracy"][-1],
            "test_acc": test_acc,
            "test_loss": test_loss,
            "time_sec": elapsed
        })
        histories[(lr, ep)] = h.history

df_results = pd.DataFrame(results)
print(df_results)

In [ ]:
pivot_acc = df_results.pivot(index="lr", columns="epochs", values="test_acc")
plt.figure(figsize=(7,5))
sns.heatmap(pivot_acc, annot=True, fmt=".3f", cmap="viridis")
plt.title("Test Accuracy: Learning Rate vs Epochs")
plt.xlabel("Epochs"); plt.ylabel("Learning Rate")
plt.show()

pivot_loss = df_results.pivot(index="lr", columns="epochs", values="test_loss")
plt.figure(figsize=(7,5))
sns.heatmap(pivot_loss, annot=True, fmt=".3f", cmap="rocket_r")
plt.title("Test Loss: Learning Rate vs Epochs")
plt.xlabel("Epochs"); plt.ylabel("Learning Rate")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
for lr in learning_rates:
    h = histories[(lr, 30)]
    plt.plot(h["val_accuracy"], label=f"lr={lr}")
plt.title("Validation Accuracy over Epochs (30 epochs) \u2014 Effect of Learning Rate")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10,6))
for lr in learning_rates:
    h = histories[(lr, 30)]
    plt.plot(h["val_loss"], label=f"lr={lr}")
plt.title("Validation Loss over Epochs (30 epochs) \u2014 Effect of Learning Rate")
plt.xlabel("Epoch"); plt.ylabel("Validation Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
tf.keras.backend.clear_session()
unstable_model = build_model(lr=1.0)  # deliberately too high
h_unstable = unstable_model.fit(x_train_flat, y_train, validation_split=0.1,
                                 epochs=10, batch_size=256, verbose=0)

plt.figure(figsize=(8,5))
plt.plot(h_unstable.history["loss"], label="train loss (lr=1.0, unstable)", color="red")
plt.plot(histories[(0.001, 15)]["loss"][:10], label="train loss (lr=0.001, stable)", color="green")
plt.title("Unstable vs Stable Learning Rate")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
tf.keras.backend.clear_session()
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01, decay_steps=1000, decay_rate=0.9)

scheduled_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])
scheduled_model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
                         loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_sched = scheduled_model.fit(x_train_flat, y_train, validation_split=0.1,
                               epochs=15, batch_size=256, verbose=0)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(histories[(0.01, 15)]["val_accuracy"], label="fixed lr=0.01")
plt.plot(h_sched.history["val_accuracy"], label="exponential decay lr (start=0.01)")
plt.title("Fixed vs Decaying Learning Rate")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Best configuration, evaluation and error analysis

In [ ]:
best_row = df_results.loc[df_results["test_acc"].idxmax()]
print("Best config:\n", best_row)

tf.keras.backend.clear_session()
best_model = build_model(lr=best_row["lr"])
best_model.fit(x_train_flat, y_train, epochs=int(best_row["epochs"]),
               batch_size=256, verbose=0)

preds = np.argmax(best_model.predict(x_test_flat, verbose=0), axis=1)

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(8,7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix (lr={best_row['lr']}, epochs={int(best_row['epochs'])})")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

print(classification_report(y_test, preds))

In [ ]:
wrong_idx = np.where(preds != y_test)[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(14,4))
for ax, idx in zip(axes.flat, wrong_idx):
    ax.imshow(x_test[idx], cmap="gray")
    ax.set_title(f"T:{y_test[idx]} P:{preds[idx]}")
    ax.axis("off")
plt.suptitle("Misclassified Examples")
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------
W1 = best_model.layers[0].get_weights()[0]  # shape (784, 128)
fig, axes = plt.subplots(4, 8, figsize=(14,7))
for i, ax in enumerate(axes.flat):
    ax.imshow(W1[:, i].reshape(28, 28), cmap="seismic")
    ax.axis("off")
plt.suptitle("Learned Weight Patterns \u2014 First Hidden Layer (32 of 128 neurons)")
plt.tight_layout()
plt.show()

In [ ]:
summary = df_results.sort_values("test_acc", ascending=False).reset_index(drop=True)
print("Ranked results (best to worst):")
print(summary.to_string(index=False))